### Advanced `Counter` Problems — Tutorial Style

In this notebook we're going to continue working with the `Counter` class from the `collections` module.

This time, instead of introducing one method after another, we'll use `Counter` to solve a collection of advanced problems.

We're going to take these problems slowly.

For most examples we'll:

1. create or inspect the data,
2. build one or more counters,
3. look at an intermediate result,
4. introduce the operation that fits the problem,
5. turn the idea into a reusable solution,
6. and test it with another example.

So the goal is not just to get the answer — it is to understand why a particular `Counter` operation is useful.

Let's start with the imports we'll need.

Most of the work will use `Counter`, but a few problems also use a `deque`, regular expressions, and `reduce`.

In [1]:
from collections import Counter, deque
from functools import reduce
import re

Before we start, recall one small but very useful behavior of counters.

If a key does not exist, its count is treated as `0`.

In [2]:
c = Counter(a=3, b=1)
c['missing']

0

That means missing keys are often painless when we compare two frequency distributions.

In [3]:
c['a'] - c['missing']

3

#### Problem 1 — Can We Build a Message From Available Characters?

Suppose we have a collection of available characters and a target message.

We want to know whether the available characters contain **enough copies of every required character**.

This is not a set problem. If the target needs three copies of `a`, then one copy of `a` is not enough.

So this is really a multiset containment problem.

Let's start with some data.

For this problem we'll ignore spaces and letter casing.

In [4]:
available_text = 'Data structures are powerful'
target_text = 'pure data'

First let's normalize the characters.

In [5]:
def normalize_characters(text):
    return [c.casefold() for c in text if c.isalnum()]

In [6]:
available_chars = normalize_characters(available_text)
target_chars = normalize_characters(target_text)

target_chars

['p', 'u', 'r', 'e', 'd', 'a', 't', 'a']

Now we can count both collections.

In [7]:
available_counter = Counter(available_chars)
target_counter = Counter(target_chars)

available_counter

Counter({'r': 4,
         'a': 3,
         't': 3,
         'u': 3,
         'e': 3,
         's': 2,
         'd': 1,
         'c': 1,
         'p': 1,
         'o': 1,
         'w': 1,
         'f': 1,
         'l': 1})

In [8]:
target_counter

Counter({'a': 2, 'p': 1, 'u': 1, 'r': 1, 'e': 1, 'd': 1, 't': 1})

What we actually care about is the **shortage**.

If we subtract available counts from required counts using Counter's binary `-`, only positive differences remain.

That is exactly what we need here.

In [9]:
missing = target_counter - available_counter
missing

Counter()

If `missing` is empty, the message can be built.

In [10]:
not missing

True

Let's turn that into a function, but instead of returning only `True` or `False`, we'll also return the missing quantities.

In [11]:
def can_build_message(available, target):
    available_counter = Counter(normalize_characters(available))
    target_counter = Counter(normalize_characters(target))
    missing = target_counter - available_counter
    return not missing, missing

In [12]:
can_build_message('aabbcc', 'abcc')

(True, Counter())

In [13]:
can_build_message('aabbcc', 'abccc')

(False, Counter({'c': 1}))

#### Problem 2 — Exact Differences Between Two Multisets

Now suppose we want a more complete comparison between two collections.

For every item, we want to know:

- what both collections share,
- what exists in excess in the first collection,
- what exists in excess in the second collection.

Let's use API response codes as an example.

In [14]:
batch_1 = [200, 200, 200, 201, 404, 404, 500]
batch_2 = [200, 200, 201, 201, 404, 503, 503]

c1 = Counter(batch_1)
c2 = Counter(batch_2)

c1, c2

(Counter({200: 3, 404: 2, 201: 1, 500: 1}),
 Counter({200: 2, 201: 2, 503: 2, 404: 1}))

The common multiplicity is the elementwise minimum.

For counters, that is `&`.

In [15]:
shared = c1 & c2
shared

Counter({200: 2, 201: 1, 404: 1})

Now let's look at what remains only in the first batch.

In [16]:
only_first = c1 - c2
only_first

Counter({200: 1, 404: 1, 500: 1})

And now the reverse direction.

In [17]:
only_second = c2 - c1
only_second

Counter({503: 2, 201: 1})

We can package all three views into a single report.

In [18]:
def compare_multisets(left, right):
    left = Counter(left)
    right = Counter(right)
    return {
        'shared': left & right,
        'only_left': left - right,
        'only_right': right - left,
    }

In [19]:
compare_multisets(batch_1, batch_2)

{'shared': Counter({200: 2, 201: 1, 404: 1}),
 'only_left': Counter({200: 1, 404: 1, 500: 1}),
 'only_right': Counter({503: 2, 201: 1})}

#### Problem 3 — Words Shared by Every Document

Suppose we have several documents and want the words that occur in **all** of them.

A normal set intersection would tell us which words occur at least once.

But a Counter intersection can tell us more: it preserves the minimum multiplicity across the documents.

In [20]:
documents = [
    'python makes data processing concise and python makes counting convenient',
    'data processing with python can make counting concise',
    'counting data with python is concise and useful',
]

We'll use a simple tokenizer that extracts word characters and normalizes case.

In [21]:
def words(text):
    return re.findall(r'\b\w+\b', text.casefold())

In [22]:
document_counters = [Counter(words(document)) for document in documents]
document_counters

[Counter({'python': 2,
          'makes': 2,
          'data': 1,
          'processing': 1,
          'concise': 1,
          'and': 1,
          'counting': 1,
          'convenient': 1}),
 Counter({'data': 1,
          'processing': 1,
          'with': 1,
          'python': 1,
          'can': 1,
          'make': 1,
          'counting': 1,
          'concise': 1}),
 Counter({'counting': 1,
          'data': 1,
          'with': 1,
          'python': 1,
          'is': 1,
          'concise': 1,
          'and': 1,
          'useful': 1})]

Let's intersect the first two counters.

In [23]:
document_counters[0] & document_counters[1]

Counter({'python': 1, 'data': 1, 'processing': 1, 'concise': 1, 'counting': 1})

And now include the third document.

In [24]:
(document_counters[0] & document_counters[1]) & document_counters[2]

Counter({'python': 1, 'data': 1, 'concise': 1, 'counting': 1})

For an arbitrary number of counters, `reduce` is a convenient way to repeat the same intersection operation.

In [25]:
common_words = reduce(lambda a, b: a & b, document_counters)
common_words

Counter({'python': 1, 'data': 1, 'concise': 1, 'counting': 1})

Compare that with a set intersection.

The set keeps membership only; the counter keeps the minimum counts as well.

In [26]:
set(words(documents[0])) & set(words(documents[1])) & set(words(documents[2]))

{'concise', 'counting', 'data', 'python'}

#### Problem 4 — Maximum Requirement Across Two Teams

Suppose two teams publish equipment requirements.

If one team requires 4 cables and another requires 7 cables, then an inventory capable of satisfying **either** team's requirement needs 7 cables.

For every item we want the maximum count.

In [27]:
engineering = Counter(battery=8, cable=4, adapter=2, flashlight=1)
operations = Counter(battery=5, cable=7, radio=2, flashlight=3)

Counter union, written with `|`, keeps the maximum positive count for each key.

In [28]:
combined_requirement = engineering | operations
combined_requirement

Counter({'battery': 8, 'cable': 7, 'flashlight': 3, 'adapter': 2, 'radio': 2})

Let's check the cable requirement manually.

In [29]:
engineering['cable'], operations['cable'], combined_requirement['cable']

(4, 7, 7)

The dual problem is the common requirement — the minimum count shared by both teams.

That's what `&` gives us.

In [30]:
engineering & operations

Counter({'battery': 5, 'cable': 4, 'flashlight': 1})

#### Problem 5 — Maximum Number of Complete Bundles

Suppose one product bundle requires several components.

How many complete bundles can we build from the current inventory?

Each required component creates its own upper bound.

In [31]:
bundle = Counter(battery=2, cable=3, charger=1, screw=4)
inventory = Counter(battery=19, cable=20, charger=8, screw=100)

Let's calculate the limit created by each component separately.

In [32]:
inventory['battery'] // bundle['battery']

9

In [33]:
inventory['cable'] // bundle['cable']

6

In [34]:
inventory['charger'] // bundle['charger']

8

In [35]:
inventory['screw'] // bundle['screw']

25

The smallest of those values is the bottleneck.

In [36]:
limits = {
    item: inventory[item] // required
    for item, required in bundle.items()
}

limits

{'battery': 9, 'cable': 6, 'charger': 8, 'screw': 25}

In [37]:
min(limits.values())

6

Let's put that logic into a function and validate the bundle definition while we're at it.

In [38]:
def complete_bundles(inventory, bundle):
    inventory = Counter(inventory)
    bundle = Counter(bundle)

    if not bundle:
        raise ValueError('bundle cannot be empty')

    if any(qty <= 0 for qty in bundle.values()):
        raise ValueError('bundle quantities must be positive')

    return min(
        inventory[item] // required
        for item, required in bundle.items()
    )

In [39]:
complete_bundles(inventory, bundle)

6

#### Problem 6 — Inventory Remaining After Building the Bundles

The previous problem told us the maximum number of bundles.

Now let's calculate how much inventory remains after building all of them.

In [40]:
number_of_bundles = complete_bundles(inventory, bundle)
number_of_bundles

6

First, create a counter for everything consumed.

In [41]:
consumed = Counter({
    item: required * number_of_bundles
    for item, required in bundle.items()
})

consumed

Counter({'screw': 24, 'cable': 18, 'battery': 12, 'charger': 6})

Now we subtract the consumed counter from the inventory.

In [42]:
remaining = inventory - consumed
remaining

Counter({'screw': 76, 'battery': 7, 'cable': 2, 'charger': 2})

Let's combine the two problems into one reusable function.

In [43]:
def build_maximum_bundles(inventory, bundle):
    inventory = Counter(inventory)
    bundle = Counter(bundle)

    count = complete_bundles(inventory, bundle)

    consumed = Counter({
        item: qty * count
        for item, qty in bundle.items()
    })

    remaining = inventory - consumed
    return count, consumed, remaining

In [44]:
build_maximum_bundles(inventory, bundle)

(6,
 Counter({'screw': 24, 'cable': 18, 'battery': 12, 'charger': 6}),
 Counter({'screw': 76, 'battery': 7, 'cable': 2, 'charger': 2}))

#### Problem 7 — Signed Event Ledger

Counters are often introduced using positive frequencies, but they can also store signed values.

Suppose we have event adjustments where some records increase a metric and others reverse or correct it.

Here negative values are meaningful, so we do **not** want to discard them.

In [45]:
events = [
    ('created', 5),
    ('processed', 4),
    ('created', 2),
    ('processed', -1),
    ('failed', 3),
    ('failed', -5),
    ('archived', 1),
]

We cannot simply write `Counter(events)` because that would count tuples.

The second value in each tuple is a weight, so we'll add it directly.

In [46]:
ledger = Counter()

for event, delta in events:
    ledger[event] += delta

ledger

Counter({'created': 7, 'processed': 3, 'archived': 1, 'failed': -2})

Notice that one balance is negative.

Unary `+` gives us a new counter containing only positive values.

In [47]:
+ledger

Counter({'created': 7, 'processed': 3, 'archived': 1})

Unary `-` flips the negative values and keeps the positive results.

That makes it a convenient way to report negative-balance magnitudes.

In [48]:
-ledger

Counter({'failed': 2})

Let's package those views into a report.

In [49]:
def signed_ledger_report(events):
    ledger = Counter()

    for event, delta in events:
        ledger[event] += delta

    return {
        'ledger': ledger,
        'positive': +ledger,
        'negative': -ledger,
        'zero': {k for k, v in ledger.items() if v == 0},
    }

In [50]:
signed_ledger_report(events)

{'ledger': Counter({'created': 7,
          'processed': 3,
          'archived': 1,
          'failed': -2}),
 'positive': Counter({'created': 7, 'processed': 3, 'archived': 1}),
 'negative': Counter({'failed': 2}),
 'zero': set()}

#### Problem 8 — Incremental Inventory Reconciliation

Suppose an inventory is affected by several independent feeds:

- receipts increase stock,
- sales reduce stock,
- damaged items reduce stock.

Instead of rebuilding the inventory from scratch after every feed, we'll update one counter incrementally.

In [51]:
starting_inventory = Counter(battery=20, cable=15, charger=8)
morning_receipts = Counter(battery=6, cable=3)
afternoon_receipts = Counter(battery=2, charger=5)
sales = Counter(battery=9, cable=7, charger=4)
damaged = Counter(battery=1, charger=2)

Let's start with a copy of the opening inventory and add both receipt batches using `update()`.

In [52]:
current = starting_inventory.copy()
current.update(morning_receipts)
current.update(afternoon_receipts)
current

Counter({'battery': 28, 'cable': 18, 'charger': 13})

Now we'll subtract sales.

We'll use `subtract()` rather than binary `-` because a negative result would be useful diagnostic information.

In [53]:
current.subtract(sales)
current

Counter({'battery': 19, 'cable': 11, 'charger': 9})

And finally subtract damaged units.

In [54]:
current.subtract(damaged)
current

Counter({'battery': 18, 'cable': 11, 'charger': 7})

Let's deliberately apply an impossible cable adjustment.

This demonstrates why keeping signed counts can be useful during reconciliation.

In [55]:
diagnostic = current.copy()
diagnostic.subtract(Counter(cable=50))
diagnostic

Counter({'battery': 18, 'charger': 7, 'cable': -39})

Now we can separate available stock from problem quantities.

In [56]:
available_inventory = +diagnostic
problem_quantities = -diagnostic

available_inventory, problem_quantities

(Counter({'battery': 18, 'charger': 7}), Counter({'cable': 39}))

#### Problem 9 — Deterministic Top-N Reporting

`most_common(n)` is usually the right tool for frequent items.

But sometimes a report requires an explicit tie-breaking rule.

We'll rank by:

1. count descending,
2. item name ascending when counts tie.

In [57]:
downloads = Counter(alpha=12, beta=9, gamma=12, delta=7, epsilon=9)
downloads.most_common()

[('alpha', 12), ('gamma', 12), ('beta', 9), ('epsilon', 9), ('delta', 7)]

The counts are correct.

Now let's make the tie rule explicit with `sorted`.

In [58]:
sorted(
    downloads.items(),
    key=lambda item: (-item[1], item[0])
)

[('alpha', 12), ('gamma', 12), ('beta', 9), ('epsilon', 9), ('delta', 7)]

And now a reusable helper.

In [59]:
def deterministic_top_n(counter, n):
    if n < 0:
        raise ValueError('n must be non-negative')

    return sorted(
        counter.items(),
        key=lambda item: (-item[1], item[0])
    )[:n]

In [60]:
deterministic_top_n(downloads, 3)

[('alpha', 12), ('gamma', 12), ('beta', 9)]

#### Problem 10 — Rolling Frequency Counts

Suppose we receive a stream of events and always want the frequencies for the **last five events**.

We could recount the whole five-element window every time.

But that would repeat work.

Instead, we'll increment the incoming event and decrement the outgoing event.

In [61]:
event_stream = [
    'read', 'read', 'write', 'read', 'delete',
    'write', 'write', 'read', 'delete', 'read'
]

window_size = 5

We'll maintain:

- a `deque` for the current window,
- a `Counter` for the window frequencies.

In [62]:
window = deque()
window_counts = Counter()
history = []

Now process the stream one event at a time.

In [63]:
for event in event_stream:
    window.append(event)
    window_counts[event] += 1

    if len(window) > window_size:
        outgoing = window.popleft()
        window_counts[outgoing] -= 1

        if window_counts[outgoing] == 0:
            del window_counts[outgoing]

    history.append((list(window), window_counts.copy()))

Let's inspect the early states.

In [64]:
history[:4]

[(['read'], Counter({'read': 1})),
 (['read', 'read'], Counter({'read': 2})),
 (['read', 'read', 'write'], Counter({'read': 2, 'write': 1})),
 (['read', 'read', 'write', 'read'], Counter({'read': 3, 'write': 1}))]

And the final state.

In [65]:
history[-1]

(['write', 'write', 'read', 'delete', 'read'],
 Counter({'read': 2, 'write': 2, 'delete': 1}))

Deleting zero-count keys is not always required, but it keeps the counter compact and makes exact equality comparisons easier to reason about.

#### Problem 11 — Finding Anagram Windows

We can reuse the rolling-counter idea for a classic string problem.

Given a text and a pattern, find every starting position where a substring is an anagram of the pattern.

In [66]:
text = 'cbaebabacd'
pattern = 'abc'

target = Counter(pattern)
target

Counter({'a': 1, 'b': 1, 'c': 1})

The window length must match the pattern length.

Let's build the first window.

In [67]:
size = len(pattern)
window_counter = Counter(text[:size])
window_counter

Counter({'c': 1, 'b': 1, 'a': 1})

The first substring is `cba`, which is an anagram of `abc`.

In [68]:
window_counter == target

True

Now slide the window.

For each move:

1. add the incoming character,
2. subtract the outgoing character,
3. delete a key if its count reaches zero,
4. compare the current counter with the target.

In [69]:
indices = []

if window_counter == target:
    indices.append(0)

for right in range(size, len(text)):
    incoming = text[right]
    outgoing = text[right - size]

    window_counter[incoming] += 1
    window_counter[outgoing] -= 1

    if window_counter[outgoing] == 0:
        del window_counter[outgoing]

    if window_counter == target:
        indices.append(right - size + 1)

indices

[0, 6]

Let's make that reusable.

In [70]:
def anagram_window_indices(text, pattern):
    if not pattern or len(pattern) > len(text):
        return []

    target = Counter(pattern)
    size = len(pattern)
    window = Counter(text[:size])
    result = []

    if window == target:
        result.append(0)

    for right in range(size, len(text)):
        incoming = text[right]
        outgoing = text[right - size]

        window[incoming] += 1
        window[outgoing] -= 1

        if window[outgoing] == 0:
            del window[outgoing]

        if window == target:
            result.append(right - size + 1)

    return result

In [71]:
anagram_window_indices('cbaebabacd', 'abc')

[0, 6]

In [72]:
anagram_window_indices('abab', 'ab')

[0, 1, 2]

#### Problem 12 — Distance Between Two Frequency Profiles

Suppose we have two counters and want a single number describing how different they are.

One simple distance is the sum of absolute count differences across all keys.

We can construct that from two positive differences:

- `A - B`,
- `B - A`.

In [73]:
profile_a = Counter(python=8, sql=4, linux=3, git=2)
profile_b = Counter(python=5, sql=6, docker=4, git=2)

In [74]:
a_excess = profile_a - profile_b
a_excess

Counter({'python': 3, 'linux': 3})

In [75]:
b_excess = profile_b - profile_a
b_excess

Counter({'docker': 4, 'sql': 2})

Now sum the excess quantities from both directions.

In [76]:
distance = sum(a_excess.values()) + sum(b_excess.values())
distance

12

Let's verify that against a direct calculation over all keys.

In [77]:
all_keys = profile_a.keys() | profile_b.keys()
manual = sum(abs(profile_a[k] - profile_b[k]) for k in all_keys)
manual

12

In [78]:
assert distance == manual

And now the reusable function.

In [79]:
def counter_distance(left, right):
    left = Counter(left)
    right = Counter(right)
    return sum((left - right).values()) + sum((right - left).values())

In [80]:
counter_distance(profile_a, profile_b)

12

#### Problem 13 — Multiset Similarity

We can also use intersection and union to define a multiset similarity score.

The numerator is the total multiplicity of `A & B`.

The denominator is the total multiplicity of `A | B`.

In [81]:
basket_a = Counter(apple=3, banana=2, orange=1)
basket_b = Counter(apple=2, banana=4, pear=2)

In [82]:
intersection = basket_a & basket_b
intersection

Counter({'apple': 2, 'banana': 2})

In [83]:
union = basket_a | basket_b
union

Counter({'banana': 4, 'apple': 3, 'pear': 2, 'orange': 1})

Now compute the totals.

In [84]:
intersection_size = sum(intersection.values())
union_size = sum(union.values())

intersection_size, union_size

(4, 10)

In [85]:
intersection_size / union_size

0.4

Let's make a function and handle the case where both counters are empty.

In [86]:
def multiset_similarity(left, right):
    left = +Counter(left)
    right = +Counter(right)

    intersection = left & right
    union = left | right

    denominator = sum(union.values())
    if denominator == 0:
        return 1.0

    return sum(intersection.values()) / denominator

In [87]:
multiset_similarity(basket_a, basket_b)

0.4

In [88]:
multiset_similarity({}, {})

1.0

#### Problem 14 — Merging Counts From Independent Workers

Suppose a large log is processed by several workers.

Each worker returns a Counter.

At the end we want one global counter.

In [89]:
worker_results = [
    Counter(ok=1200, warning=14, error=5),
    Counter(ok=980, warning=19, error=7),
    Counter(ok=1105, warning=11, timeout=3),
    Counter(ok=1250, error=4, timeout=1),
]

The most direct streaming-friendly solution is to start with an empty counter and repeatedly call `update()`.

In [90]:
global_counts = Counter()

for worker_counter in worker_results:
    global_counts.update(worker_counter)

global_counts

Counter({'ok': 4535, 'warning': 44, 'error': 16, 'timeout': 4})

For a small in-memory list, repeated counter addition also works.

In [91]:
sum(worker_results, Counter())

Counter({'ok': 4535, 'warning': 44, 'error': 16, 'timeout': 4})

In [92]:
assert global_counts == sum(worker_results, Counter())

#### Problem 15 — Applying Per-Item Limits

Suppose a system receives requested quantities, but every item has a maximum allowed quantity.

For each key we want the smaller of:

- requested quantity,
- allowed limit.

That is an elementwise minimum — exactly what `&` does.

In [93]:
requested = Counter(cpu=12, memory=30, disk=8, gpu=5)
limits = Counter(cpu=10, memory=16, disk=20, gpu=2)

In [94]:
accepted = requested & limits
accepted

Counter({'memory': 16, 'cpu': 10, 'disk': 8, 'gpu': 2})

Now calculate what was rejected because it exceeded the limits.

In [95]:
rejected = requested - accepted
rejected

Counter({'memory': 14, 'gpu': 3, 'cpu': 2})

Let's turn that into a helper.

In [96]:
def apply_limits(requested, limits):
    requested = +Counter(requested)
    limits = +Counter(limits)

    accepted = requested & limits
    rejected = requested - accepted
    return accepted, rejected

In [97]:
apply_limits(requested, limits)

(Counter({'memory': 16, 'cpu': 10, 'disk': 8, 'gpu': 2}),
 Counter({'memory': 14, 'gpu': 3, 'cpu': 2}))

#### Problem 16 — Reconstructing Repeated Values With `elements()`

A Counter can be viewed as a compressed representation of repeated values.

For example, `Counter(red=3, blue=2)` tells us that `red` occurs three times and `blue` occurs twice.

In [98]:
compressed = Counter(red=3, blue=2, green=1)
list(compressed.elements())

['red', 'red', 'red', 'blue', 'blue', 'green']

Let's use this to create a small weighted work queue.

In [99]:
priority_weights = Counter(critical=4, high=3, medium=2, low=1)
queue = list(priority_weights.elements())
queue

['critical',
 'critical',
 'critical',
 'critical',
 'high',
 'high',
 'high',
 'medium',
 'medium',
 'low']

This is fine for small counts.

But remember that `elements()` produces one value per unit of positive count.

So a very large count can expand into a very large iterable.

In [100]:
large = Counter(event=10_000_000)
large

Counter({'event': 10000000})

If we only need frequency arithmetic, it is usually better to leave the data compressed in the Counter rather than materializing millions of repeated values.

#### Problem 17 — First Unique Item

Sometimes frequency alone is not enough.

Suppose we need the **first** character that occurs exactly once.

The Counter tells us which characters have frequency `1`, while the original string preserves the order.

In [101]:
text = 'statistics'
counts = Counter(text)
counts

Counter({'s': 3, 't': 3, 'i': 2, 'a': 1, 'c': 1})

Let's see which characters are unique.

In [102]:
[ch for ch, count in counts.items() if count == 1]

['a', 'c']

Now scan the original text and stop at the first character whose count is `1`.

In [103]:
for ch in text:
    if counts[ch] == 1:
        first_unique = ch
        break
else:
    first_unique = None

first_unique

'a'

Let's make that reusable.

In [104]:
def first_unique_item(sequence):
    counts = Counter(sequence)

    for item in sequence:
        if counts[item] == 1:
            return item

    return None

In [105]:
first_unique_item('swiss')

'w'

In [106]:
first_unique_item('aabbcc')

#### Problem 18 — End-to-End Warehouse Reconciliation

Let's finish with a larger example.

Suppose a warehouse has:

- opening inventory,
- deliveries,
- customer shipments,
- customer returns,
- damaged units,
- a final physical count.

We want to calculate the expected closing inventory and then compare it with what was actually counted.

In [107]:
opening = Counter(battery=50, cable=40, charger=25, case=30, mouse=20)
deliveries = Counter(battery=20, cable=10, charger=12, mouse=8)
shipments = Counter(battery=31, cable=22, charger=18, case=9, mouse=11)
returns = Counter(battery=3, cable=2, case=1)
damaged = Counter(battery=2, charger=1, mouse=3)
physical_count = Counter(battery=40, cable=30, charger=18, case=22, mouse=13)

We'll calculate the expected inventory one step at a time.

Start from a copy of the opening balance.

In [108]:
expected = opening.copy()
expected

Counter({'battery': 50, 'cable': 40, 'case': 30, 'charger': 25, 'mouse': 20})

Deliveries increase stock.

In [109]:
expected.update(deliveries)
expected

Counter({'battery': 70, 'cable': 50, 'charger': 37, 'case': 30, 'mouse': 28})

Shipments reduce stock.

We'll use `subtract()` because a negative result would be useful evidence of inconsistent data.

In [110]:
expected.subtract(shipments)
expected

Counter({'battery': 39, 'cable': 28, 'case': 21, 'charger': 19, 'mouse': 17})

Returns increase stock again.

In [111]:
expected.update(returns)
expected

Counter({'battery': 42, 'cable': 30, 'case': 22, 'charger': 19, 'mouse': 17})

Damaged units reduce sellable stock.

In [112]:
expected.subtract(damaged)
expected

Counter({'battery': 40, 'cable': 30, 'case': 22, 'charger': 18, 'mouse': 14})

Now let's compare the physical count with the expected count.

We'll calculate:

`physical - expected`

Positive results are overages; negative results are shortages.

In [113]:
difference = physical_count.copy()
difference.subtract(expected)
difference

Counter({'battery': 0, 'cable': 0, 'charger': 0, 'case': 0, 'mouse': -1})

Unary `+` gives the overages.

In [114]:
overages = +difference
overages

Counter()

Unary `-` gives the shortage magnitudes.

In [115]:
shortages = -difference
shortages

Counter({'mouse': 1})

Let's now write the whole reconciliation as a function.

In [116]:
def warehouse_reconciliation(opening, deliveries, shipments, returns, damaged, physical_count):
    expected = Counter(opening)
    expected.update(deliveries)
    expected.subtract(shipments)
    expected.update(returns)
    expected.subtract(damaged)

    difference = Counter(physical_count)
    difference.subtract(expected)

    return {
        'expected': expected,
        'physical': Counter(physical_count),
        'difference': difference,
        'overages': +difference,
        'shortages': -difference,
    }

In [117]:
report = warehouse_reconciliation(
    opening,
    deliveries,
    shipments,
    returns,
    damaged,
    physical_count,
)

report

{'expected': Counter({'battery': 40,
          'cable': 30,
          'case': 22,
          'charger': 18,
          'mouse': 14}),
 'physical': Counter({'battery': 40,
          'cable': 30,
          'case': 22,
          'charger': 18,
          'mouse': 13}),
 'difference': Counter({'battery': 0,
          'cable': 0,
          'charger': 0,
          'case': 0,
          'mouse': -1}),
 'overages': Counter(),
 'shortages': Counter({'mouse': 1})}

We can also calculate the total number of units that need investigation.

In [118]:
units_to_investigate = (
    sum(report['overages'].values())
    + sum(report['shortages'].values())
)

units_to_investigate

1

##### Alternate Solution Without Counter

Just for comparison, let's calculate the final physical-versus-expected difference using a plain dictionary.

This is useful because it shows some of the bookkeeping that Counter performs for us.

In [119]:
expected_dict = dict(report['expected'])
physical_dict = dict(physical_count)

all_items = set(expected_dict) | set(physical_dict)

difference_dict = {
    item: physical_dict.get(item, 0) - expected_dict.get(item, 0)
    for item in all_items
}

difference_dict

{'case': 0, 'cable': 0, 'charger': 0, 'mouse': -1, 'battery': 0}

The dictionary solution works.

But notice what we had to handle ourselves:

- collect all possible keys,
- use `.get(key, 0)` for missing values,
- manually split positive and negative differences if we want separate reports.

Counter gives those ideas much more directly.

#### Final Checks

Let's run a few assertions covering the important reusable functions from the notebook.

In [120]:
assert can_build_message('aabbcc', 'abcc') == (True, Counter())
assert can_build_message('aabbcc', 'abccc')[0] is False

assert complete_bundles(Counter(a=10, b=7), Counter(a=2, b=3)) == 2

assert anagram_window_indices('cbaebabacd', 'abc') == [0, 6]
assert anagram_window_indices('abab', 'ab') == [0, 1, 2]

assert counter_distance(Counter(a=3, b=1), Counter(a=1, c=2)) == 5

assert apply_limits(
    Counter(a=10, b=2),
    Counter(a=4, b=5),
) == (Counter(a=4, b=2), Counter(a=6))

assert first_unique_item('swiss') == 'w'

print('All checks passed.')

All checks passed.


At this point we've used `Counter` for much more than simple frequency counting.

We've used it for:

- multiset containment,
- missing-item reports,
- multiset differences,
- intersections across many counters,
- per-key maxima and minima,
- bundle calculations,
- signed ledgers,
- incremental reconciliation,
- deterministic reporting,
- rolling windows,
- anagram detection,
- distance and similarity measures,
- merging independent worker results,
- per-item caps,
- repeated iteration,
- order-sensitive queries combined with frequency data,
- and a complete warehouse audit workflow.

The key idea is that a Counter is not just a convenient dictionary for counting.

It is also a compact representation of a **multiset**, and many problems become much easier once we recognize that structure.